# Jewelry Category Classifier — Training Notebook

**10-class supervised classifier** trained on precomputed CLIP + DINOv2 embeddings.  
Replaces zero-shot CLIP classification in the indexing/search pipeline.

### Notebook Sections
0. Install & Imports  
1. **Configuration** — all paths in one place  
2. Mount Drive & verify paths  
3. Load metadata CSV & explore  
4. **Generate embeddings** (checkpointed every 500 images → crash-safe)  
5. Load / verify embeddings  
6. Stratified train / val / test split  
7. **Train & evaluate** all 9 (model × embedding) combinations  
8. Detailed analysis of winning model  
9. Export winning classifier for production  

> **Resumability:** Every heavy step writes checkpoints to Drive.  
> Re-running a cell after a disconnect picks up where it left off.

## 0. Install & Imports

In [ ]:
# ── Install dependencies (only needed once per runtime) ──────────────────────
!pip install -q sentence-transformers xgboost scikit-learn imbalanced-learn \
    matplotlib seaborn pandas numpy tqdm joblib pillow

import os, sys, gc, time, base64, json, warnings, logging
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

**Edit this cell only.** Every path the notebook uses is defined here.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        CONFIGURE ALL PATHS HERE                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ---- Google Drive mount point ------------------------------------------------
DRIVE_MOUNT_POINT = Path("/content/drive")

# ---- Data paths (metadata CSV + images live in a separate folder from repo) ---
METADATA_CSV = DRIVE_MOUNT_POINT / "MyDrive" / "RFID-Project" / "csv" / "rebuilt_labelled_metadata_new.csv"
IMAGES_ROOT  = DRIVE_MOUNT_POINT / "MyDrive" / "RFID-Project"

# ---- Column names in the metadata CSV ----------------------------------------
COL_IMAGE_PATH = "image_path"       # column with relative image path (relative to IMAGES_ROOT)
COL_CATEGORY   = "category_label"   # column with human-verified category label

# ---- Repo path (your cloned codebase with engines/, preprocess/, etc.) -------
REPO_ROOT = DRIVE_MOUNT_POINT / "MyDrive" / "RFID-Project" / "Jewelry-Search-Prod--feat-Inference-pipeline"

# ---- Model paths (for CLIP SentenceTransformer & DINOv2) ---------------------
#   BiRefNet: local folder with the HF model files
#   CLIP: local folder OR HuggingFace hub name (auto-downloads)
#   DINOv2: torch.hub key — one of dinov2_vitl14, dinov2_vitb14, dinov2_vits14
MODELS_ROOT      = DRIVE_MOUNT_POINT / "MyDrive" / "RFID-Project" / "models"
BIREFNET_MODEL_PATH = str(MODELS_ROOT / "BiRefNet")
CLIP_MODEL_PATH     = str(MODELS_ROOT / "clip-ViT-L-14")
DINOV2_MODEL_KEY    = "dinov2_vitl14"       # 1024-dim

# ---- Output directory (all checkpoints, embeddings, models saved here) -------
OUTPUT_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "RFID-Project" / "classifier_outputs"

# ---- Derived paths (don't edit) ----------------------------------------------
EMBEDDINGS_CSV      = OUTPUT_DIR / "embeddings_checkpoint.csv"   # intermediate CSV with embeddings
EMBEDDINGS_NPZ      = OUTPUT_DIR / "embeddings_arrays.npz"      # numpy arrays (fast loading)
SPLIT_CSV           = OUTPUT_DIR / "train_val_test_split.csv"    # stratified split assignments
RESULTS_DIR         = OUTPUT_DIR / "results"                     # confusion matrices, reports
MODELS_DIR          = OUTPUT_DIR / "models"                      # trained model artifacts

# ---- Hyperparameters ---------------------------------------------------------
CHECKPOINT_EVERY    = 500        # save embeddings every N images
RANDOM_SEED         = 42
BATCH_SIZE_CLIP     = 16        # images per GPU batch for CLIP
BATCH_SIZE_DINO     = 16        # images per GPU batch for DINOv2

# ---- Target classes (10 total, order determines label encoding) ---------------
TARGET_CLASSES = [
    "earring", "necklace", "bracelet", "pendant", "bangle",
    "ring", "mangalsutra", "chain", "anklet", "other"
]

print("Configuration loaded.")

## 2. Mount Drive & Verify Paths

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
except ImportError:
    print("Not running in Colab — skipping drive mount.")

# ── Create output directories ─────────────────────────────────────────────────
for d in [OUTPUT_DIR, RESULTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  ✓ {d}")

# ── Add repo to sys.path so we can import engines/preprocess ──────────────────
repo_str = str(REPO_ROOT)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
    print(f"  ✓ Added to sys.path: {repo_str}")

# ── Verify key paths exist ────────────────────────────────────────────────────
assert METADATA_CSV.exists(), f"Metadata CSV not found: {METADATA_CSV}"
assert IMAGES_ROOT.exists(),  f"Images root not found: {IMAGES_ROOT}"
assert REPO_ROOT.exists(),    f"Repo root not found: {REPO_ROOT}"
print("\nAll paths verified. ✓")

## 3. Load Metadata CSV & Explore

In [ ]:
# ── Load metadata ─────────────────────────────────────────────────────────────
df_meta = pd.read_csv(METADATA_CSV)
print(f"Total rows in CSV: {len(df_meta)}")
print(f"Columns: {list(df_meta.columns)}")
df_meta.head(3)

In [ ]:
# ── Filter to target classes only (exclude paper/unknown if present) ──────────
df = df_meta[df_meta[COL_CATEGORY].isin(TARGET_CLASSES)].copy()
df = df.reset_index(drop=True)
print(f"Rows after filtering to {len(TARGET_CLASSES)} target classes: {len(df)}")
print()

# ── Class distribution ────────────────────────────────────────────────────────
class_counts = df[COL_CATEGORY].value_counts()
print("Class distribution:")
print(class_counts.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
class_counts.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Class Distribution (10 jewelry categories)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Verify a few image paths actually exist on disk ───────────────────────────
sample_paths = df[COL_IMAGE_PATH].sample(5, random_state=RANDOM_SEED)
for p in sample_paths:
    full = IMAGES_ROOT / p
    status = '✓' if full.exists() else '✗ MISSING'
    print(f"  {status}  {full}")

## 4. Generate Embeddings (Crash-Safe with Checkpoints)

Uses the **existing engine code** from the repo (`CLIPEngine`, `DINOv2Engine`, `ImageProcessor`).  
Saves progress every 500 images to `EMBEDDINGS_CSV` + `EMBEDDINGS_NPZ`.  
**Re-running after a disconnect resumes from the last checkpoint.**

In [ ]:
# ── Build a lightweight config object for the engines ─────────────────────────
@dataclass
class EngineConfig:
    device: str = "auto"
    birefnet_model_path: str = BIREFNET_MODEL_PATH
    clip_model_path: str = CLIP_MODEL_PATH
    dinov2_model: str = DINOV2_MODEL_KEY
    use_fp16: bool = False

engine_cfg = EngineConfig()

# ── Import and initialize engines from repo ───────────────────────────────────
from engines.birefnet_engine import BiRefNetEngine
from engines.clip_engine import CLIPEngine
from engines.dinov2_engine import DINOv2Engine
from preprocess.image_processor import ImageProcessor

print("Loading BiRefNet engine (background removal)...")
birefnet_engine = BiRefNetEngine(engine_cfg)

print("\nLoading CLIP engine...")
clip_engine = CLIPEngine(engine_cfg)

print("\nLoading DINOv2 engine...")
dino_engine = DINOv2Engine(engine_cfg)

print("\nInitializing image preprocessor...")
img_processor = ImageProcessor(crop_padding=2, crop_alpha_threshold=30)

print("\n✓ All engines ready (BiRefNet + CLIP + DINOv2).")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EMBEDDING GENERATION — resumable, checkpointed every CHECKPOINT_EVERY images
# ══════════════════════════════════════════════════════════════════════════════

def load_existing_progress():
    """
    Load already-computed embeddings from the checkpoint files.
    Returns (set_of_done_paths, list_of_clip_embs, list_of_dino_embs, list_of_records)
    """
    done_paths = set()
    clip_embs, dino_embs, records = [], [], []

    if EMBEDDINGS_CSV.exists() and EMBEDDINGS_NPZ.exists():
        df_done = pd.read_csv(EMBEDDINGS_CSV)
        npz = np.load(EMBEDDINGS_NPZ)
        clip_arr = npz["clip_embeddings"]
        dino_arr = npz["dino_embeddings"]

        assert len(df_done) == len(clip_arr) == len(dino_arr), \
            f"Checkpoint mismatch: CSV={len(df_done)}, clip={len(clip_arr)}, dino={len(dino_arr)}"

        done_paths = set(df_done["image_path"].tolist())
        clip_embs = list(clip_arr)
        dino_embs = list(dino_arr)
        records = df_done.to_dict('records')
        print(f"  ↻ Resumed: {len(done_paths)} images already embedded.")
    else:
        print("  Starting fresh — no existing checkpoint found.")

    return done_paths, clip_embs, dino_embs, records


def save_checkpoint(records, clip_embs, dino_embs):
    """Save current progress to Drive (CSV index + NPZ arrays)."""
    df_out = pd.DataFrame(records)
    df_out.to_csv(EMBEDDINGS_CSV, index=False)
    np.savez_compressed(
        EMBEDDINGS_NPZ,
        clip_embeddings=np.array(clip_embs, dtype=np.float32),
        dino_embeddings=np.array(dino_embs, dtype=np.float32),
    )


def generate_single_embedding(image_path_rel):
    """
    Generate CLIP (768-d) and DINOv2 (1024-d) embeddings for one image.
    Full pipeline matching production:
      1. Load RGB image
      2. BiRefNet → RGBA (background removal)
      3. ImageProcessor:
         - CLIP path:   RGBA → white-bg composite (uncropped) → CLIP
         - DINOv2 path: RGBA → crop to subject → DINOv2 (resize/center-crop)
    Returns (clip_emb, dino_emb) both as numpy arrays.
    """
    full_path = IMAGES_ROOT / image_path_rel
    img = Image.open(full_path).convert("RGB")

    # Step 1: Background removal → RGBA
    rgba = birefnet_engine.get_rgba(img)

    # Step 2: Preprocess — produces white_bg (for CLIP) and cropped (for DINOv2)
    processed = img_processor.process(rgba)

    # CLIP embedding from white_bg (uncropped)
    # Resize to 224x224 for CLIP (SentenceTransformer handles this internally)
    clip_emb = clip_engine.run(processed.white_bg)       # (768,)

    # DINOv2 embedding from cropped image
    # DINOv2Engine._transform handles resize(256) + center_crop(224)
    dino_emb = dino_engine.run(processed.cropped)         # (1024,)

    return clip_emb, dino_emb


# ── Main embedding loop ───────────────────────────────────────────────────────
done_paths, clip_embs, dino_embs, records = load_existing_progress()
remaining = df[~df[COL_IMAGE_PATH].isin(done_paths)].reset_index(drop=True)
print(f"  Remaining to process: {len(remaining)} / {len(df)} total")

errors = []
t0 = time.time()

for i, row in tqdm(remaining.iterrows(), total=len(remaining), desc="Embedding"):
    img_path = row[COL_IMAGE_PATH]
    category = row[COL_CATEGORY]

    try:
        clip_emb, dino_emb = generate_single_embedding(img_path)
        clip_embs.append(clip_emb)
        dino_embs.append(dino_emb)
        records.append({
            "image_path": img_path,
            "category": category,
            "status": "ok",
        })
    except Exception as e:
        errors.append((img_path, str(e)))
        # Insert zero vectors so indices stay aligned
        clip_embs.append(np.zeros(clip_engine.embedding_dim, dtype=np.float32))
        dino_embs.append(np.zeros(dino_engine.embedding_dim, dtype=np.float32))
        records.append({
            "image_path": img_path,
            "category": category,
            "status": f"error: {e}",
        })

    # ── Checkpoint ────────────────────────────────────────────────────────────
    processed_in_this_run = i + 1
    if processed_in_this_run % CHECKPOINT_EVERY == 0:
        save_checkpoint(records, clip_embs, dino_embs)
        elapsed = time.time() - t0
        rate = processed_in_this_run / elapsed
        tqdm.write(
            f"    💾 Checkpoint @ {len(records)} total "
            f"({rate:.1f} img/s, {len(errors)} errors)"
        )

# ── Final save ─────────────────────────────────────────────────────────────────
save_checkpoint(records, clip_embs, dino_embs)
elapsed = time.time() - t0
print(f"\n✓ Done. {len(records)} total embeddings in {elapsed:.0f}s.")
if errors:
    print(f"  ⚠ {len(errors)} errors:")
    for p, e in errors[:10]:
        print(f"    {p}: {e}")

In [ ]:
# ── Free GPU memory (engines no longer needed) ────────────────────────────────
del birefnet_engine, clip_engine, dino_engine, img_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory freed.")

## 5. Load & Verify Embeddings

From here on, everything runs on CPU (pure sklearn/xgboost).  
**You can restart the runtime and re-run from this cell** — no GPU needed.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RESUME POINT — run from here if runtime disconnected after embeddings done
# ══════════════════════════════════════════════════════════════════════════════
# (Re-run Section 0 imports + Section 1 config first, then jump here)

# ── Load checkpoint files ─────────────────────────────────────────────────────
df_emb = pd.read_csv(EMBEDDINGS_CSV)
npz = np.load(EMBEDDINGS_NPZ)
clip_all = npz["clip_embeddings"]    # (N, 768)
dino_all = npz["dino_embeddings"]    # (N, 1024)

print(f"Loaded {len(df_emb)} records")
print(f"  CLIP  shape: {clip_all.shape}")
print(f"  DINOv2 shape: {dino_all.shape}")

# ── Drop error rows ───────────────────────────────────────────────────────────
ok_mask = df_emb["status"] == "ok"
print(f"  OK: {ok_mask.sum()}, Errors: {(~ok_mask).sum()}")

df_emb = df_emb[ok_mask].reset_index(drop=True)
clip_all = clip_all[ok_mask.values]
dino_all = dino_all[ok_mask.values]

categories = df_emb["category"].values
print(f"\nFinal dataset: {len(df_emb)} images, {clip_all.shape[1]}+{dino_all.shape[1]} dims")
print(pd.Series(categories).value_counts().to_string())

In [ ]:
# ── L2-normalize embeddings ───────────────────────────────────────────────────
def l2_normalize(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-8)
    return X / norms

clip_normed = l2_normalize(clip_all)
dino_normed = l2_normalize(dino_all)
concat_normed = np.hstack([clip_normed, dino_normed])  # (N, 1792)

print(f"CLIP normed:   {clip_normed.shape} — sample norms: {np.linalg.norm(clip_normed[:3], axis=1)}")
print(f"DINOv2 normed: {dino_normed.shape} — sample norms: {np.linalg.norm(dino_normed[:3], axis=1)}")
print(f"Concat:        {concat_normed.shape}")

## 6. Stratified Train / Val / Test Split

In [ ]:
# ── Encode labels ─────────────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(TARGET_CLASSES)    # fixed order
y = le.transform(categories)
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ── Stratified split: 70/15/15 ────────────────────────────────────────────────
if SPLIT_CSV.exists():
    print(f"\n↻ Loading existing split from {SPLIT_CSV}")
    df_split = pd.read_csv(SPLIT_CSV)
    # Merge split assignments back
    df_emb["split"] = df_split["split"]
else:
    # First split: 70% train, 30% temp
    idx_train, idx_temp, y_train, y_temp = train_test_split(
        np.arange(len(y)), y,
        test_size=0.30, stratify=y, random_state=RANDOM_SEED
    )
    # Second split: 50/50 of temp → 15% val, 15% test
    idx_val, idx_test, _, _ = train_test_split(
        idx_temp, y_temp,
        test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED
    )

    df_emb["split"] = "train"
    df_emb.loc[idx_val, "split"] = "val"
    df_emb.loc[idx_test, "split"] = "test"

    # Save split for reproducibility
    df_emb[["image_path", "category", "split"]].to_csv(SPLIT_CSV, index=False)
    print(f"  ✓ Split saved to {SPLIT_CSV}")

# ── Summary ───────────────────────────────────────────────────────────────────
for s in ["train", "val", "test"]:
    mask = df_emb["split"] == s
    print(f"  {s:5s}: {mask.sum():5d}  ({mask.sum()/len(df_emb)*100:.1f}%)")

# ── Build index masks ─────────────────────────────────────────────────────────
train_mask = (df_emb["split"] == "train").values
val_mask   = (df_emb["split"] == "val").values
test_mask  = (df_emb["split"] == "test").values

y_train = y[train_mask]
y_val   = y[val_mask]
y_test  = y[test_mask]

print(f"\nVal class dist:  {dict(Counter(le.inverse_transform(y_val)))}")
print(f"Test class dist: {dict(Counter(le.inverse_transform(y_test)))}")

## 7. Prepare Feature Matrices

In [ ]:
# ── Build feature dicts for the 3 embedding configs ───────────────────────────
embedding_configs = {
    "CLIP-768": {
        "train": clip_normed[train_mask],
        "val":   clip_normed[val_mask],
        "test":  clip_normed[test_mask],
    },
    "DINOv2-1024": {
        "train": dino_normed[train_mask],
        "val":   dino_normed[val_mask],
        "test":  dino_normed[test_mask],
    },
    "CLIP+DINO-1792": {
        "train": concat_normed[train_mask],
        "val":   concat_normed[val_mask],
        "test":  concat_normed[test_mask],
    },
}

for name, splits in embedding_configs.items():
    print(f"  {name:18s}  train={splits['train'].shape}  val={splits['val'].shape}  test={splits['test'].shape}")

## 8. Train & Evaluate All 9 Combinations

3 models × 3 embedding configs = 9 runs.  
Each model uses `class_weight='balanced'` (or equivalent) for imbalance handling.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING — all 9 (model × embedding) combinations
# ══════════════════════════════════════════════════════════════════════════════

def compute_sample_weights(y_arr):
    """Inverse-frequency class weights for XGBoost (which doesn't take class_weight dict)."""
    counts = Counter(y_arr)
    n = len(y_arr)
    n_classes = len(counts)
    weights = np.array([
        n / (n_classes * counts[c]) for c in y_arr
    ], dtype=np.float32)
    return weights


def train_logistic_regression(X_train, y_train, X_val, y_val):
    model = LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        solver='lbfgs',
        multi_class='multinomial',
        C=1.0,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def train_random_forest(X_train, y_train, X_val, y_val):
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        class_weight='balanced',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def train_xgboost(X_train, y_train, X_val, y_val):
    sw_train = compute_sample_weights(y_train)
    sw_val   = compute_sample_weights(y_val)

    dtrain = xgb.DMatrix(X_train, label=y_train, weight=sw_train)
    dval   = xgb.DMatrix(X_val, label=y_val, weight=sw_val)

    params = {
        'objective': 'multi:softprob',
        'num_class': len(TARGET_CLASSES),
        'eval_metric': 'mlogloss',
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': RANDOM_SEED,
        'verbosity': 0,
        'tree_method': 'hist',
    }

    model = xgb.train(
        params, dtrain,
        num_boost_round=500,
        evals=[(dval, 'val')],
        early_stopping_rounds=30,
        verbose_eval=False,
    )
    return model


def predict_model(model, X):
    """Unified prediction for sklearn models and xgboost."""
    if isinstance(model, xgb.Booster):
        dmat = xgb.DMatrix(X)
        probs = model.predict(dmat)  # (N, num_class)
        return probs.argmax(axis=1), probs
    else:
        preds = model.predict(X)
        probs = model.predict_proba(X)
        return preds, probs


# ── Run all 9 combinations ────────────────────────────────────────────────────
model_factories = {
    "LogisticRegression": train_logistic_regression,
    "XGBoost":           train_xgboost,
    "RandomForest":      train_random_forest,
}

results = []       # list of dicts for the summary table
trained_models = {} # key = (model_name, emb_name) → model object

for emb_name, splits in embedding_configs.items():
    X_tr, X_va, X_te = splits["train"], splits["val"], splits["test"]

    for model_name, factory in model_factories.items():
        key = (model_name, emb_name)
        print(f"\n{'='*70}")
        print(f"  Training: {model_name} + {emb_name}")
        print(f"{'='*70}")

        t0 = time.time()
        model = factory(X_tr, y_train, X_va, y_val)
        train_time = time.time() - t0

        # ── Evaluate on val ───────────────────────────────────────────────────
        y_pred_val, y_prob_val = predict_model(model, X_va)
        val_macro_f1 = f1_score(y_val, y_pred_val, average='macro')
        val_acc = accuracy_score(y_val, y_pred_val)

        # ── Evaluate on test ──────────────────────────────────────────────────
        y_pred_test, y_prob_test = predict_model(model, X_te)
        test_macro_f1 = f1_score(y_test, y_pred_test, average='macro')
        test_acc = accuracy_score(y_test, y_pred_test)

        # ── Per-class report (val) ────────────────────────────────────────────
        print(f"\n  Val Macro-F1: {val_macro_f1:.4f}  |  Val Acc: {val_acc:.4f}  |  Time: {train_time:.1f}s")
        print(classification_report(
            y_val, y_pred_val,
            target_names=le.classes_, digits=3, zero_division=0
        ))

        trained_models[key] = model
        results.append({
            "model": model_name,
            "embedding": emb_name,
            "val_macro_f1": round(val_macro_f1, 4),
            "val_accuracy": round(val_acc, 4),
            "test_macro_f1": round(test_macro_f1, 4),
            "test_accuracy": round(test_acc, 4),
            "train_time_s": round(train_time, 1),
        })

print("\n" + "="*70)
print("  ALL 9 RUNS COMPLETE")
print("="*70)

## 9. Results Summary

In [ ]:
# ── Summary table sorted by val macro-F1 ─────────────────────────────────────
df_results = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
df_results.index = range(1, len(df_results) + 1)
df_results.index.name = "rank"

print(df_results.to_string())

# Save to CSV
df_results.to_csv(RESULTS_DIR / "bakeoff_results.csv")
print(f"\n  ✓ Saved to {RESULTS_DIR / 'bakeoff_results.csv'}")

## 10. Detailed Analysis of the Winning Model

In [ ]:
# ── Identify winner ───────────────────────────────────────────────────────────
winner = df_results.iloc[0]
winner_key = (winner["model"], winner["embedding"])
print(f"🏆 Winner: {winner_key[0]} + {winner_key[1]}")
print(f"   Val  Macro-F1: {winner['val_macro_f1']:.4f}  |  Acc: {winner['val_accuracy']:.4f}")
print(f"   Test Macro-F1: {winner['test_macro_f1']:.4f}  |  Acc: {winner['test_accuracy']:.4f}")

best_model = trained_models[winner_key]
best_emb_name = winner["embedding"]
X_te_best = embedding_configs[best_emb_name]["test"]

y_pred_best, y_prob_best = predict_model(best_model, X_te_best)

# ── Full classification report on TEST set ────────────────────────────────────
print("\n" + "="*70)
print("  TEST SET — Classification Report")
print("="*70)
print(classification_report(
    y_test, y_pred_best,
    target_names=le.classes_, digits=3, zero_division=0
))

# ── Focus on 'other' and 'anklet' recall ──────────────────────────────────────
report_dict = classification_report(
    y_test, y_pred_best,
    target_names=le.classes_, output_dict=True, zero_division=0
)
for cls in ["other", "anklet"]:
    r = report_dict[cls]
    print(f"  {cls:12s}  precision={r['precision']:.3f}  recall={r['recall']:.3f}  f1={r['f1-score']:.3f}  support={int(r['support'])}")

In [ ]:
# ── Confusion matrix (test set) ──────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f'{title} — Counts')
    axes[0].set_ylabel('True')
    axes[0].set_xlabel('Predicted')

    # Percentages
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title(f'{title} — Row-Normalized %')
    axes[1].set_ylabel('True')
    axes[1].set_xlabel('Predicted')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  ✓ Saved: {save_path}")
    plt.show()


plot_confusion_matrix(
    y_test, y_pred_best, le.classes_,
    title=f"Winner: {winner_key[0]} + {winner_key[1]}",
    save_path=RESULTS_DIR / "confusion_matrix_winner_test.png"
)

In [ ]:
# ── Confusion matrices for ALL 9 runs (val set) ──────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(24, 20))

for idx, row in df_results.iterrows():
    i = idx - 1  # rank is 1-based
    ax = axes[i // 3][i % 3]

    key = (row["model"], row["embedding"])
    model = trained_models[key]
    X_va = embedding_configs[row["embedding"]]["val"]

    y_pred_v, _ = predict_model(model, X_va)
    cm = confusion_matrix(y_val, y_pred_v)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm_pct, annot=True, fmt='.0f', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax,
                cbar=False, annot_kws={'size': 7})
    ax.set_title(f"#{idx} {row['model']}\n{row['embedding']} (F1={row['val_macro_f1']:.3f})", fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_ylabel('True' if i % 3 == 0 else '')
    ax.set_xlabel('Pred')

plt.suptitle('All 9 Runs — Val Confusion Matrices (row-normalized %)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices_all_9.png", dpi=150, bbox_inches='tight')
plt.show()

## 11. (Optional) SMOTE for Minority Classes

Only run this if `other` / `anklet` recall is weak after class-weighting.  
Applies SMOTE in embedding space (valid since embeddings are continuous vectors).

In [ ]:
# ── Check if SMOTE is warranted ──────────────────────────────────────────────
other_recall  = report_dict["other"]["recall"]
anklet_recall = report_dict["anklet"]["recall"]
SMOTE_THRESHOLD = 0.70  # only apply if recall < this

needs_smote = (other_recall < SMOTE_THRESHOLD) or (anklet_recall < SMOTE_THRESHOLD)
print(f"  other  recall = {other_recall:.3f}  {'⚠ below threshold' if other_recall < SMOTE_THRESHOLD else '✓ OK'}")
print(f"  anklet recall = {anklet_recall:.3f}  {'⚠ below threshold' if anklet_recall < SMOTE_THRESHOLD else '✓ OK'}")

if needs_smote:
    print("\n  → Applying SMOTE on winning embedding config...")
    from imblearn.over_sampling import SMOTE

    X_tr_best = embedding_configs[best_emb_name]["train"]
    sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=3)
    X_tr_smote, y_tr_smote = sm.fit_resample(X_tr_best, y_train)
    print(f"  Before SMOTE: {len(X_tr_best)}  →  After: {len(X_tr_smote)}")
    print(f"  New class dist: {dict(Counter(y_tr_smote))}")

    # Retrain the winning model type with SMOTE data
    factory = model_factories[winner_key[0]]
    smote_model = factory(X_tr_smote, y_tr_smote,
                          embedding_configs[best_emb_name]["val"], y_val)

    y_pred_smote, _ = predict_model(smote_model, X_te_best)
    smote_f1 = f1_score(y_test, y_pred_smote, average='macro')
    original_f1 = winner['test_macro_f1']

    print(f"\n  Test Macro-F1:  original={original_f1:.4f}  SMOTE={smote_f1:.4f}")
    print(classification_report(
        y_test, y_pred_smote,
        target_names=le.classes_, digits=3, zero_division=0
    ))

    report_smote = classification_report(
        y_test, y_pred_smote,
        target_names=le.classes_, output_dict=True, zero_division=0
    )
    print(f"  other  recall: {other_recall:.3f} → {report_smote['other']['recall']:.3f}")
    print(f"  anklet recall: {anklet_recall:.3f} → {report_smote['anklet']['recall']:.3f}")

    # Use SMOTE model if it improved
    if smote_f1 > original_f1:
        print("\n  ✓ SMOTE improved results — using SMOTE model as final.")
        best_model = smote_model
        # Update confusion matrix
        plot_confusion_matrix(
            y_test, y_pred_smote, le.classes_,
            title=f"Winner + SMOTE: {winner_key[0]} + {winner_key[1]}",
            save_path=RESULTS_DIR / "confusion_matrix_winner_smote.png"
        )
    else:
        print("\n  ✗ SMOTE did not improve — keeping original model.")
else:
    print("\n  ✓ Class-weighting is sufficient. Skipping SMOTE.")

## 12. Export Winning Classifier for Production

Exports the winning model + label encoder + metadata in a format ready  
to drop into the indexing/search pipeline, preserving `category` and  
`category_confidence` fields.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPORT — production-ready classifier package
# ══════════════════════════════════════════════════════════════════════════════

export_meta = {
    "model_type": winner_key[0],
    "embedding_config": winner_key[1],
    "target_classes": TARGET_CLASSES,
    "label_mapping": {int(k): v for k, v in enumerate(le.classes_)},
    "val_macro_f1": float(winner["val_macro_f1"]),
    "test_macro_f1": float(winner["test_macro_f1"]),
    "embedding_dim": int(embedding_configs[best_emb_name]["train"].shape[1]),
    "clip_dim": 768,
    "dino_dim": 1024,
    "needs_l2_norm": True,
    "random_seed": RANDOM_SEED,
    "smote_applied": needs_smote and (f1_score(y_test, predict_model(best_model, X_te_best)[0], average='macro') > winner['test_macro_f1']) if needs_smote else False,
}

# ── Save model ────────────────────────────────────────────────────────────────
if isinstance(best_model, xgb.Booster):
    model_path = MODELS_DIR / "category_classifier.xgb"
    best_model.save_model(str(model_path))
else:
    model_path = MODELS_DIR / "category_classifier.joblib"
    joblib.dump(best_model, model_path)

# ── Save label encoder ────────────────────────────────────────────────────────
le_path = MODELS_DIR / "label_encoder.joblib"
joblib.dump(le, le_path)

# ── Save metadata ─────────────────────────────────────────────────────────────
meta_path = MODELS_DIR / "model_metadata.json"
with open(meta_path, 'w') as f:
    json.dump(export_meta, f, indent=2)

print(f"\n✓ Exported to {MODELS_DIR}:")
for p in sorted(MODELS_DIR.glob("*")):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:40s} {size_kb:8.1f} KB")

print(f"\nModel metadata:")
print(json.dumps(export_meta, indent=2))

## 13. Drop-in Replacement: `SupervisedCategoryClassifier`

This class has the same `.classify()` interface as the existing  
`CategoryClassifier`, so it can be swapped in without changing pipeline code.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DROP-IN REPLACEMENT — same interface as classifiers/category_classifier.py
# ══════════════════════════════════════════════════════════════════════════════

CLASSIFIER_CODE = '''
"""
classifiers/supervised_category_classifier.py
Supervised 10-class jewelry category classifier.
Drop-in replacement for CategoryClassifier (zero-shot CLIP).

Trained on precomputed CLIP + DINOv2 embeddings.
Same .classify() interface: returns (category, confidence, scores_dict).
"""
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import joblib

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False


class SupervisedCategoryClassifier:
    """
    Supervised replacement for zero-shot CategoryClassifier.

    Usage:
        clf = SupervisedCategoryClassifier(model_dir="path/to/models/")
        category, confidence, all_scores = clf.classify(
            clip_embedding,          # (768,) raw CLIP embedding
            dino_embedding=None,     # (1024,) raw DINOv2 embedding (if needed)
        )
    """

    def __init__(self, model_dir: str | Path, threshold: float = 0.20):
        model_dir = Path(model_dir)
        self.threshold = threshold

        # Load metadata
        with open(model_dir / "model_metadata.json") as f:
            self.meta = json.load(f)

        # Load label encoder
        self.le = joblib.load(model_dir / "label_encoder.joblib")

        # Load model
        self.model_type = self.meta["model_type"]
        self.embedding_config = self.meta["embedding_config"]

        if self.model_type == "XGBoost":
            assert HAS_XGB, "xgboost is required for this model"
            self.model = xgb.Booster()
            self.model.load_model(str(model_dir / "category_classifier.xgb"))
        else:
            self.model = joblib.load(model_dir / "category_classifier.joblib")

        print(f"  [SupervisedCategoryClassifier] Ready — "
              f"{self.model_type} + {self.embedding_config}, "
              f"{len(self.le.classes_)} classes")

    def _build_features(self, clip_embedding: np.ndarray,
                        dino_embedding: np.ndarray | None) -> np.ndarray:
        """Build feature vector based on the winning embedding config."""
        clip_norm = clip_embedding / (np.linalg.norm(clip_embedding) + 1e-8)

        if "CLIP+DINO" in self.embedding_config:
            assert dino_embedding is not None, \
                "DINOv2 embedding required for CLIP+DINO config"
            dino_norm = dino_embedding / (np.linalg.norm(dino_embedding) + 1e-8)
            return np.concatenate([clip_norm, dino_norm])

        elif "DINOv2" in self.embedding_config:
            assert dino_embedding is not None, \
                "DINOv2 embedding required for DINOv2 config"
            dino_norm = dino_embedding / (np.linalg.norm(dino_embedding) + 1e-8)
            return dino_norm

        else:  # CLIP only
            return clip_norm

    def classify(self, clip_embedding: np.ndarray,
                 dino_embedding: np.ndarray | None = None) -> tuple:
        """
        Classify a jewelry image.

        Args:
            clip_embedding: (768,) raw CLIP embedding (un-normalized OK).
            dino_embedding: (1024,) raw DINOv2 embedding (needed if config uses it).

        Returns:
            (category_str, confidence_float, {category: score} dict)
            Same interface as the old CategoryClassifier.
        """
        features = self._build_features(clip_embedding, dino_embedding)
        X = features.reshape(1, -1)

        if self.model_type == "XGBoost":
            dmat = xgb.DMatrix(X)
            probs = self.model.predict(dmat)[0]  # (num_class,)
        else:
            probs = self.model.predict_proba(X)[0]

        scores = {self.le.classes_[i]: float(probs[i])
                  for i in range(len(probs))}

        best_idx = int(np.argmax(probs))
        best_cat = self.le.classes_[best_idx]
        best_conf = float(probs[best_idx])

        if best_conf < self.threshold:
            return "unknown", best_conf, scores

        return best_cat, best_conf, scores
'''

# Write the classifier file to the output directory
clf_path = MODELS_DIR / "supervised_category_classifier.py"
with open(clf_path, 'w') as f:
    f.write(CLASSIFIER_CODE.strip())

print(f"✓ Drop-in classifier written to: {clf_path}")
print(f"\nTo use in the pipeline, replace:")
print(f"  from classifiers.category_classifier import CategoryClassifier")
print(f"with:")
print(f"  from classifiers.supervised_category_classifier import SupervisedCategoryClassifier")
print(f"  clf = SupervisedCategoryClassifier(model_dir='{MODELS_DIR}')")

## 14. Final Summary

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║               JEWELRY CATEGORY CLASSIFIER — SUMMARY               ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  Dataset:      {len(df_emb):,} images, {len(TARGET_CLASSES)} classes")
print(f"║  Winner:       {winner_key[0]} + {winner_key[1]}")
print(f"║  Val  F1:      {winner['val_macro_f1']:.4f}")
print(f"║  Test F1:      {winner['test_macro_f1']:.4f}")
print("║")
print(f"║  Outputs saved to: {OUTPUT_DIR}")
print("║")
print("║  Files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(OUTPUT_DIR)
        size_kb = p.stat().st_size / 1024
        print(f"║    {str(rel):45s} {size_kb:8.1f} KB")
print("╚══════════════════════════════════════════════════════════════════════╝")